
# GVH Diagonal Cubic `.28.21.2.1.2.4` — FAST
## Exact Open Coupling-Neighborhood Reduction Regularity

### Unique lock

The parent checkpoint proved strong hyperbolicity on one fixed healthy anisotropic witness:

\[
(a_0,a_1,a_2,a_3)
=
\left(
\frac34,-\frac15,-\frac14,-\frac3{10}
\right),
\]

\[
(K_S,\kappa_D,M_{\rm Pl}^2)=(1,2,1).
\]

This notebook does **not** yet claim strong hyperbolicity on a parameter domain.

Its only task is to prove that the Hamilton-Dirac/principal reduction remains algebraically regular on an explicit open neighbourhood of the coupling point while the anisotropic background is held fixed.

We certify the closed rational box

\[
\boxed{
|K_S-1|\le\frac1{100},\qquad
|\kappa_D-2|\le\frac1{100},\qquad
|M_{\rm Pl}^2-1|\le\frac1{100}.
}
\]

Its interior is therefore an explicit nonempty open coupling neighbourhood.

Required locks:

1. the six structural principal null directions remain exact;
2. the kinetic rank remains exactly \(14\);
3. the fixed \(14+6\) complement remains invertible;
4. the principal Noether chain remains exact;
5. the global gauge-coordinate atlas remains regular.

No spectral-root or strong-hyperbolicity statement away from the central witness is promoted here.


In [1]:

from __future__ import annotations

import sys, json
from pathlib import Path

import sympy as sp

PARENT_B3 = {
    "artifact":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.3_"
        "Analytic_Uniform_Directional_Projector_and_Bounded_Diagonalizer_Certificate_FAST(1).ipynb",
    "executed_size_bytes": 63693,
    "executed_sha256": '692d40c9978b2eb45ac2475b6c656e591770ba9d1616acfffa6db3fcd44ac6b8',
    "machine_clean": True,
    "strong_hyperbolicity_proven_fixed_healthy_witness": True,
    "strong_hyperbolicity_global_parameter_domain_proven": False,
}

G28212124_PROVENANCE_GATE_PASS = all([
    PARENT_B3["machine_clean"],
    PARENT_B3["strong_hyperbolicity_proven_fixed_healthy_witness"],
    not PARENT_B3["strong_hyperbolicity_global_parameter_domain_proven"],
])

assert G28212124_PROVENANCE_GATE_PASS

print("Python =",sys.version.split()[0])
print("SymPy =",sp.__version__)
print("G28212124_PROVENANCE_GATE_PASS =",G28212124_PROVENANCE_GATE_PASS)


Python = 3.13.15
SymPy = 1.14.0
G28212124_PROVENANCE_GATE_PASS = True



# 1. Inherited exact raw principal construction

The next two code cells are copied from the canonical B3 source.

They reconstruct the raw quadratic principal pencil directly from the candidate action and retain:

\[
D_i=\frac12M_i.
\]

No frozen coupling values are substituted yet.


In [2]:

eta=sp.diag(-1,1,1,1)

a0,a1,a2=sp.symbols("a0 a1 a2",real=True)
a3=-a0-a1-a2

Abar=sp.diag(a0,a1,a2,a3)
Qbar=sp.factor(sp.trace(Abar*Abar))

KS,kappaD,Mpl2=sp.symbols(
    "K_S kappa_D Mpl2",
    real=True
)

names=[
    "n","beta1","beta2","beta3",
    "h11","h22","h33","h12","h13","h23",
    "D00","D01","D02","D03",
    "D11","D22","D33","D12","D13","D23",
]

h_basis=[]
D_basis=[]

for name in names:
    h=sp.zeros(4)
    d=sp.zeros(4)

    if name=="n":
        h[0,0]=-2
    elif name.startswith("beta"):
        i=int(name[-1])
        h[0,i]=h[i,0]=1
    elif name.startswith("h"):
        ij=name[1:]
        i,j=int(ij[0]),int(ij[1])
        h[i,j]=h[j,i]=1
    elif name.startswith("D"):
        ij=name[1:]
        i,j=int(ij[0]),int(ij[1])
        d[i,j]=d[j,i]=1

    h_basis.append(h)
    D_basis.append(d)

dA_basis=[]

for h,d in zip(h_basis,D_basis):
    dM=-eta*h*Abar+eta*d
    dA=dM-sp.trace(dM)*sp.eye(4)/4
    dA_basis.append(dA)

def build_principal_matrix(p):
    H=sp.zeros(20)

    # Pure-GVH-P sector
    for alpha_idx in range(4):
        B=[]
        J=[]
        C=[]

        for h,dA in zip(h_basis,dA_basis):
            Gamma=sp.zeros(4)

            for mu in range(4):
                for nu in range(4):
                    acc=0
                    for rho in range(4):
                        acc += eta[mu,rho]*(
                            p[alpha_idx]*h[rho,nu]
                            +p[nu]*h[rho,alpha_idx]
                            -p[rho]*h[alpha_idx,nu]
                        )/2
                    Gamma[mu,nu]=acc

            Bj=p[alpha_idx]*dA+Gamma*Abar-Abar*Gamma
            B.append(Bj)
            J.append(sp.trace(Abar*Bj))
            C.append(Abar*Bj-Bj*Abar)

        sign=eta[alpha_idx,alpha_idx]

        for j in range(20):
            for k in range(j,20):
                shape=Qbar*sp.trace(B[j]*B[k])-J[j]*J[k]
                angle=sp.trace(C[j]*C[k])

                val=(
                    -KS*sign*shape
                    -kappaD*sp.Rational(1,2)*sign*angle
                )

                H[j,k]+=val
                if k!=j:
                    H[k,j]+=val

    # Einstein-Hilbert / Fierz-Pauli benchmark
    pvec=sp.Matrix(p)
    pup=eta*pvec
    p2=(pvec.T*eta*pvec)[0]

    attrs=[]

    for h in h_basis[:10]:
        hup=eta*h*eta
        v=[
            sum(p[mu]*hup[mu,nu] for mu in range(4))
            for nu in range(4)
        ]
        w=[
            sum(pup[lam]*h[lam,nu] for lam in range(4))
            for nu in range(4)
        ]
        trh=sp.trace(eta*h)
        vp=sum(v[nu]*p[nu] for nu in range(4))
        attrs.append((h,hup,v,w,trh,vp))

    for j in range(10):
        hj,hjup,vj,wj,trj,vpj=attrs[j]

        for k in range(j,10):
            hk,hkup,vk,wk,trk,vpk=attrs[k]

            inner=sum(
                hj[mu,nu]*hkup[mu,nu]
                for mu in range(4)
                for nu in range(4)
            )

            BF=(
                p2*inner
                -sum(
                    vj[nu]*wk[nu]+vk[nu]*wj[nu]
                    for nu in range(4)
                )
                +vpj*trk
                +vpk*trj
                -p2*trj*trk
            )

            val=-Mpl2*sp.Rational(1,4)*BF

            H[j,k]+=val
            if k!=j:
                H[k,j]+=val

    return H

e0=(1,0,0,0)
e1=(0,1,0,0)
e2=(0,0,1,0)
e3=(0,0,0,1)

P_e0=build_principal_matrix(e0)
P_e1=build_principal_matrix(e1)
P_e2=build_principal_matrix(e2)
P_e3=build_principal_matrix(e3)

K_raw=P_e0

M_raw={
    1:build_principal_matrix((1,1,0,0))-P_e0-P_e1,
    2:build_principal_matrix((1,0,1,0))-P_e0-P_e2,
    3:build_principal_matrix((1,0,0,1))-P_e0-P_e3,
}

G_raw={
    (1,1):P_e1,
    (2,2):P_e2,
    (3,3):P_e3,
    (1,2):(build_principal_matrix((0,1,1,0))-P_e1-P_e2)/2,
    (1,3):(build_principal_matrix((0,1,0,1))-P_e1-P_e3)/2,
    (2,3):(build_principal_matrix((0,0,1,1))-P_e2-P_e3)/2,
}

G2821161_RAW_PENCIL_RECONSTRUCTED=all([
    K_raw.shape==(20,20),
    all(M_raw[i].shape==(20,20) for i in (1,2,3)),
    all(G_raw[key].shape==(20,20) for key in G_raw),
])

assert G2821161_RAW_PENCIL_RECONSTRUCTED

print("G2821161_RAW_PENCIL_RECONSTRUCTED =",G2821161_RAW_PENCIL_RECONSTRUCTED)


G2821161_RAW_PENCIL_RECONSTRUCTED = True


In [3]:
D_raw={i:sp.simplify(M_raw[i]/2) for i in (1,2,3)}
assert all(sp.simplify(D_raw[i]+D_raw[i].T-M_raw[i])==sp.zeros(20) for i in (1,2,3))
print("Principal Legendre representative D_i=M_i/2 fixed")

Principal Legendre representative D_i=M_i/2 fixed



# 2. Fix only the background, leave the three couplings symbolic

We now freeze:

\[
a_0=\frac34,\qquad
a_1=-\frac15,\qquad
a_2=-\frac14,\qquad
a_3=-\frac3{10},
\]

but keep:

\[
K_S,\qquad \kappa_D,\qquad M_{\rm Pl}^2
\]

symbolic.

The central coupling point remains:

\[
c_\star=(1,2,1).
\]


In [4]:

background_subs={
    a0:sp.Rational(3,4),
    a1:-sp.Rational(1,5),
    a2:-sp.Rational(1,4),
}

coupling_center={
    KS:sp.Integer(1),
    kappaD:sp.Integer(2),
    Mpl2:sp.Integer(1),
}

delta=sp.Rational(1,100)

K_cpl=sp.simplify(K_raw.subs(background_subs))
M_cpl={
    i:sp.simplify(M_raw[i].subs(background_subs))
    for i in (1,2,3)
}
D_cpl={
    i:sp.simplify(D_raw[i].subs(background_subs))
    for i in (1,2,3)
}
G_cpl={
    key:sp.simplify(G_raw[key].subs(background_subs))
    for key in G_raw
}

G28212124_RAW_PENCIL_SYMMETRY_STRUCTURAL_PASS=all([
    K_cpl==K_cpl.T,
    all(M_cpl[i]==M_cpl[i].T for i in (1,2,3)),
    all(G_cpl[key]==G_cpl[key].T for key in G_cpl),
])

assert G28212124_RAW_PENCIL_SYMMETRY_STRUCTURAL_PASS

print("delta =",delta)
print(
    "G28212124_RAW_PENCIL_SYMMETRY_STRUCTURAL_PASS =",
    G28212124_RAW_PENCIL_SYMMETRY_STRUCTURAL_PASS
)


delta = 1/100
G28212124_RAW_PENCIL_SYMMETRY_STRUCTURAL_PASS = True



# 3. Exact six-dimensional structural null sector

For the fixed anisotropic background, construct:

- four diffeomorphism principal directions;
- one trace-shift direction;
- one radial nondynamical direction.

The crucial identity is required **symbolically in all three couplings**:

\[
\boxed{
K(K_S,\kappa_D,M_{\rm Pl}^2)\,N_6=0.
}
\]

Thus:

\[
\operatorname{rank}K\le 14
\]

throughout coupling space.


In [5]:

a3_fixed=sp.simplify(
    (-a0-a1-a2).subs(background_subs)
)

a_fixed=[
    background_subs[a0],
    background_subs[a1],
    background_subs[a2],
    a3_fixed,
]

def original_gauge_vectors_fixed_background(p):
    vectors=[]

    for sigma in range(4):
        zeta=[0,0,0,0]
        zeta[sigma]=1

        hg=sp.zeros(4)
        dD=sp.zeros(4)

        for mu in range(4):
            for nu in range(4):
                hg[mu,nu]=(
                    p[mu]*zeta[nu]
                    +p[nu]*zeta[mu]
                )

                dD[mu,nu]=(
                    a_fixed[nu]*p[mu]*zeta[nu]
                    +a_fixed[mu]*p[nu]*zeta[mu]
                )

        vec=sp.zeros(20,1)

        vec[0]=-hg[0,0]/2
        vec[1]=hg[0,1]
        vec[2]=hg[0,2]
        vec[3]=hg[0,3]
        vec[4]=hg[1,1]
        vec[5]=hg[2,2]
        vec[6]=hg[3,3]
        vec[7]=hg[1,2]
        vec[8]=hg[1,3]
        vec[9]=hg[2,3]

        vals=[
            dD[0,0],dD[0,1],dD[0,2],dD[0,3],
            dD[1,1],dD[2,2],dD[3,3],
            dD[1,2],dD[1,3],dD[2,3],
        ]

        for j,val in enumerate(vals,start=10):
            vec[j]=val

        vectors.append(vec)

    trace=sp.zeros(20,1)
    trace[10]=-1
    trace[14]=1
    trace[15]=1
    trace[16]=1

    vectors.append(trace)

    return vectors

N_diff=sp.Matrix.hstack(
    *original_gauge_vectors_fixed_background(
        (1,0,0,0)
    )[:4]
)

N_trace=sp.zeros(20,1)
N_trace[10]=-1
N_trace[14]=1
N_trace[15]=1
N_trace[16]=1

N_radial=sp.zeros(20,1)
N_radial[10]=-a_fixed[0]
N_radial[14]=a_fixed[1]
N_radial[15]=a_fixed[2]
N_radial[16]=a_fixed[3]

N6=sp.Matrix.hstack(
    N_diff,
    N_trace,
    N_radial,
)

G28212124_STRUCTURAL_KERNEL_PASS=all([
    N6.shape==(20,6),
    N6.rank()==6,
    sp.simplify(K_cpl*N6)==sp.zeros(20,6),
])

assert G28212124_STRUCTURAL_KERNEL_PASS

print("rank N6 =",N6.rank())
print(
    "G28212124_STRUCTURAL_KERNEL_PASS =",
    G28212124_STRUCTURAL_KERNEL_PASS
)


rank N6 = 6
G28212124_STRUCTURAL_KERNEL_PASS = True



# 4. Fixed complement from the canonical witness

At the central coupling point, choose the 14-dimensional kinetic image:

\[
R_\star=\operatorname{colspace}K(c_\star).
\]

Then:

\[
T_\star=(R_\star,N_6).
\]

We require:

\[
\boxed{
\det T_\star\neq0.
}
\]

Because \(T_\star\) is coupling-independent, this gives a fixed configuration-space decomposition throughout the coupling box.

The variable reduced kinetic matrix is:

\[
K_{14}(c)=R_\star^T K(c)R_\star.
\]


In [6]:

K_center=K_cpl.subs(coupling_center)

assert K_center.rank()==14

R_ref=sp.Matrix.hstack(
    *K_center.columnspace()
)

T_ref=sp.Matrix.hstack(
    R_ref,
    N6,
)

K14_cpl=sp.simplify(
    R_ref.T*K_cpl*R_ref
)

G28212124_FIXED_COMPLEMENT_PASS=all([
    R_ref.shape==(20,14),
    R_ref.rank()==14,
    T_ref.shape==(20,20),
    T_ref.rank()==20,
])

assert G28212124_FIXED_COMPLEMENT_PASS

print("rank K_center =",K_center.rank())
print("rank R_ref =",R_ref.rank())
print("rank T_ref =",T_ref.rank())
print(
    "G28212124_FIXED_COMPLEMENT_PASS =",
    G28212124_FIXED_COMPLEMENT_PASS
)


rank K_center = 14
rank R_ref = 14
rank T_ref = 20
G28212124_FIXED_COMPLEMENT_PASS = True



# 5. Exact factorisation of the reduced kinetic determinant

For the fixed background, the exact determinant factorises into a nonzero rational constant times:

\[
K_S^2
\]

and nine explicit coupling factors.

Eight are affine and one is cubic.

The closed coupling box will be certified by proving every factor stays strictly away from zero.


In [7]:

det_K14_cpl=sp.factor(
    K14_cpl.det(method="domain-ge")
)

linear_factors=[
    151*KS-100*kappaD,
    151*KS-kappaD,
    604*KS-441*kappaD,
    604*KS-361*kappaD,
    604*KS-kappaD,
    242204*KS+160000*Mpl2-162401*kappaD,
    459644*KS+320000*Mpl2-290321*kappaD,
    507964*KS+320000*Mpl2-354481*kappaD,
]

cubic_factor=(
    8769939874416*KS**3
    -17452761140808*KS**2*kappaD
    -11616128000000*KS*Mpl2**2
    +11538912519351*KS*kappaD**2
    -5120000000000*Mpl2**3
    +7756832000000*Mpl2**2*kappaD
    -2534495840100*kappaD**3
)

expected_nonconstant=(
    KS**2
    *(151*KS-100*kappaD)
    *(151*KS-kappaD)
    *(604*KS-441*kappaD)
    *(604*KS-361*kappaD)
    *(604*KS-kappaD)**2
    *(242204*KS+160000*Mpl2-162401*kappaD)
    *(459644*KS+320000*Mpl2-290321*kappaD)
    *(507964*KS+320000*Mpl2-354481*kappaD)
    *cubic_factor
)

det_constant=sp.simplify(
    det_K14_cpl/expected_nonconstant
)

G28212124_KINETIC_DETERMINANT_FACTORIZATION_PASS=all([
    det_constant.is_Rational,
    det_constant!=0,
    sp.simplify(
        det_K14_cpl
        -det_constant*expected_nonconstant
    )==0,
])

assert G28212124_KINETIC_DETERMINANT_FACTORIZATION_PASS

print("determinant constant nonzero =",det_constant!=0)
print(
    "G28212124_KINETIC_DETERMINANT_FACTORIZATION_PASS =",
    G28212124_KINETIC_DETERMINANT_FACTORIZATION_PASS
)


determinant constant nonzero = True
G28212124_KINETIC_DETERMINANT_FACTORIZATION_PASS = True



# 6. Exact interval margins for all affine factors

For an affine factor:

\[
L(c)=L(c_\star)+
\sum_i \ell_i\,\delta c_i,
\]

with:

\[
|\delta c_i|\le\delta=\frac1{100},
\]

we have the exact bound:

\[
|L(c)-L(c_\star)|
\le
\delta\sum_i|\ell_i|.
\]

A factor is certified nonzero on the entire closed box when:

\[
|L(c_\star)|
-
\delta\sum_i|\ell_i|
>0.
\]


In [8]:

coupling_vars=[
    KS,
    kappaD,
    Mpl2,
]

linear_margin_ledger=[]

for factor in linear_factors:
    center_value=sp.expand(
        factor
    ).subs(coupling_center)

    variation_bound=sp.simplify(
        delta*sum(
            abs(sp.diff(factor,var))
            for var in coupling_vars
        )
    )

    safe_margin=sp.simplify(
        abs(center_value)-variation_bound
    )

    linear_margin_ledger.append({
        "factor":str(factor),
        "center_value":str(center_value),
        "variation_bound":str(variation_bound),
        "safe_margin":str(safe_margin),
        "certified":bool(safe_margin>0),
    })

G28212124_ALL_AFFINE_FACTOR_MARGINS_PASS=all(
    row["certified"]
    for row in linear_margin_ledger
)

# KS itself:
KS_positive_margin=sp.Rational(99,100)

G28212124_KS_NONZERO_BOX_PASS=(
    KS_positive_margin>0
)

assert G28212124_ALL_AFFINE_FACTOR_MARGINS_PASS
assert G28212124_KS_NONZERO_BOX_PASS

for row in linear_margin_ledger:
    print(row)

print("KS lower bound =",KS_positive_margin)
print(
    "G28212124_ALL_AFFINE_FACTOR_MARGINS_PASS =",
    G28212124_ALL_AFFINE_FACTOR_MARGINS_PASS
)


{'factor': '151*K_S - 100*kappa_D', 'center_value': '-49', 'variation_bound': '251/100', 'safe_margin': '4649/100', 'certified': True}
{'factor': '151*K_S - kappa_D', 'center_value': '149', 'variation_bound': '38/25', 'safe_margin': '3687/25', 'certified': True}
{'factor': '604*K_S - 441*kappa_D', 'center_value': '-278', 'variation_bound': '209/20', 'safe_margin': '5351/20', 'certified': True}
{'factor': '604*K_S - 361*kappa_D', 'center_value': '-118', 'variation_bound': '193/20', 'safe_margin': '2167/20', 'certified': True}
{'factor': '604*K_S - kappa_D', 'center_value': '602', 'variation_bound': '121/20', 'safe_margin': '11919/20', 'certified': True}
{'factor': '242204*K_S + 160000*Mpl2 - 162401*kappa_D', 'center_value': '77402', 'variation_bound': '112921/20', 'safe_margin': '1435119/20', 'certified': True}
{'factor': '459644*K_S + 320000*Mpl2 - 290321*kappa_D', 'center_value': '199002', 'variation_bound': '213993/20', 'safe_margin': '3766047/20', 'certified': True}
{'factor': '5079


# 7. Exact cubic-factor margin

Introduce deviations:

\[
x=K_S-1,\qquad
y=\kappa_D-2,\qquad
z=M_{\rm Pl}^2-1.
\]

For:

\[
|x|,|y|,|z|\le\frac1{100},
\]

expand:

\[
Q(c)=Q(c_\star)+\Delta Q(x,y,z).
\]

Using the coefficientwise triangle inequality gives an exact rational upper bound:

\[
|\Delta Q|\le B_Q.
\]

The sign is frozen throughout the closed box if:

\[
|Q(c_\star)|-B_Q>0.
\]


In [9]:

dx,dy,dz=sp.symbols(
    "dx dy dz",
    real=True,
)

cubic_shifted=sp.expand(
    cubic_factor.subs({
        KS:1+dx,
        kappaD:2+dy,
        Mpl2:1+dz,
    })
)

cubic_center=sp.expand(
    cubic_shifted.subs({
        dx:0,
        dy:0,
        dz:0,
    })
)

cubic_remainder=sp.expand(
    cubic_shifted-cubic_center
)

P_cubic_remainder=sp.Poly(
    cubic_remainder,
    dx,
    dy,
    dz,
    domain=sp.QQ,
)

cubic_variation_bound=sp.simplify(
    sum(
        abs(coeff)
        *delta**sum(monom)
        for monom,coeff
        in P_cubic_remainder.terms()
    )
)

cubic_safe_margin=sp.simplify(
    abs(cubic_center)
    -cubic_variation_bound
)

G28212124_CUBIC_FACTOR_MARGIN_PASS=all([
    cubic_center<0,
    cubic_variation_bound>0,
    cubic_safe_margin>0,
])

assert G28212124_CUBIC_FACTOR_MARGIN_PASS

print("cubic center =",cubic_center)
print(
    "cubic variation bound =",
    cubic_variation_bound
)
print("cubic safe margin =",cubic_safe_margin)
print(
    "G28212124_CUBIC_FACTOR_MARGIN_PASS =",
    G28212124_CUBIC_FACTOR_MARGIN_PASS
)


cubic center = -1478363050596
cubic variation bound = 9326441357867007/40000
cubic safe margin = 49808080665972993/40000
G28212124_CUBIC_FACTOR_MARGIN_PASS = True



# 8. Exact kinetic-rank theorem on the coupling box

Every factor in:

\[
\det K_{14}(c)
\]

is now certified nonzero on the full closed box.

Hence:

\[
\det K_{14}(c)\neq0
\]

everywhere there, and therefore:

\[
\operatorname{rank}K(c)\ge14.
\]

But the exact six-dimensional structural kernel already gave:

\[
\operatorname{rank}K(c)\le14.
\]

Thus:

\[
\boxed{
\operatorname{rank}K(c)=14
}
\]

uniformly throughout the box.


In [10]:

G28212124_KINETIC_RANK14_CLOSED_BOX_CERTIFIED=all([
    G28212124_STRUCTURAL_KERNEL_PASS,
    G28212124_FIXED_COMPLEMENT_PASS,
    G28212124_KINETIC_DETERMINANT_FACTORIZATION_PASS,
    G28212124_KS_NONZERO_BOX_PASS,
    G28212124_ALL_AFFINE_FACTOR_MARGINS_PASS,
    G28212124_CUBIC_FACTOR_MARGIN_PASS,
])

assert G28212124_KINETIC_RANK14_CLOSED_BOX_CERTIFIED

print(
    "G28212124_KINETIC_RANK14_CLOSED_BOX_CERTIFIED =",
    G28212124_KINETIC_RANK14_CLOSED_BOX_CERTIFIED
)


G28212124_KINETIC_RANK14_CLOSED_BOX_CERTIFIED = True



# 9. Principal Noether chain across the full coupling family

Let:

\[
B(\mathbf n)=n_iM_i,
\]

\[
C(\mathbf n)=n_in_jG_{ij}.
\]

For the fixed background but arbitrary couplings, verify exactly:

\[
KG_1=0,
\]

\[
KG_0+BG_1=0,
\]

\[
BG_0+CG_1=0,
\]

\[
CG_0=0.
\]

These identities show that opening the coupling box does not break the principal diffeomorphism chain.


In [11]:

n1,n2,n3=sp.symbols(
    "n1 n2 n3",
    real=True,
)

B_cpl=(
    n1*M_cpl[1]
    +n2*M_cpl[2]
    +n3*M_cpl[3]
)

C_cpl=(
    n1**2*G_cpl[(1,1)]
    +n2**2*G_cpl[(2,2)]
    +n3**2*G_cpl[(3,3)]
    +2*n1*n2*G_cpl[(1,2)]
    +2*n1*n3*G_cpl[(1,3)]
    +2*n2*n3*G_cpl[(2,3)]
)

G0_symbolic=sp.Matrix.hstack(
    *original_gauge_vectors_fixed_background(
        (0,n1,n2,n3)
    )[:4]
)

G1_symbolic=N_diff

noether_checks=[
    sp.simplify(
        K_cpl*G1_symbolic
    )==sp.zeros(20,4),

    sp.simplify(
        K_cpl*G0_symbolic
        +B_cpl*G1_symbolic
    )==sp.zeros(20,4),

    sp.simplify(
        B_cpl*G0_symbolic
        +C_cpl*G1_symbolic
    )==sp.zeros(20,4),

    sp.simplify(
        C_cpl*G0_symbolic
    )==sp.zeros(20,4),
]

G28212124_PRINCIPAL_NOETHER_CHAIN_COUPLING_FAMILY_PASS=all(
    noether_checks
)

assert G28212124_PRINCIPAL_NOETHER_CHAIN_COUPLING_FAMILY_PASS

print("Noether checks =",noether_checks)
print(
    "G28212124_PRINCIPAL_NOETHER_CHAIN_COUPLING_FAMILY_PASS =",
    G28212124_PRINCIPAL_NOETHER_CHAIN_COUPLING_FAMILY_PASS
)


Noether checks = [True, True, True, True]
G28212124_PRINCIPAL_NOETHER_CHAIN_COUPLING_FAMILY_PASS = True



# 10. Global gauge-coordinate atlas remains regular

Because the background and the fixed complement \(T_\star\) do not vary with the couplings, the coordinate matrix of the spatial diffeomorphism directions is coupling-independent.

Define:

\[
Q(\mathbf n)
=
\left[
T_\star^{-1}G_0(\mathbf n)
\right]_{1:14}.
\]

Its Gram determinant is an even polynomial in the direction components.

We require every coefficient to be strictly positive.

Therefore:

\[
\boxed{
\operatorname{rank}Q(\mathbf n)=4
}
\]

for every nonzero direction and for every coupling in the box.


In [12]:

T_ref_inv=T_ref.inv()

Q_symbolic=sp.simplify(
    (T_ref_inv*G0_symbolic)[:14,:]
)

Gram_global=sp.simplify(
    Q_symbolic.T*Q_symbolic
)

det_Gram=sp.factor(
    Gram_global.det()
)

poly_Gram=sp.Poly(
    sp.expand(det_Gram),
    n1,
    n2,
    n3,
)

gram_terms=poly_Gram.terms()

gram_even_exponents=all(
    all(e%2==0 for e in monom)
    for monom,coeff in gram_terms
)

gram_positive_coeffs=all(
    coeff>0
    for monom,coeff in gram_terms
)

G28212124_GLOBAL_GAUGE_COORDINATE_ATLAS_PASS=all([
    Q_symbolic.shape==(14,4),
    gram_even_exponents,
    gram_positive_coeffs,
    len(gram_terms)>0,
])

assert G28212124_GLOBAL_GAUGE_COORDINATE_ATLAS_PASS

print("det Gram total degree =",poly_Gram.total_degree())
print("det Gram term count =",len(gram_terms))
print("all exponents even =",gram_even_exponents)
print("all coefficients positive =",gram_positive_coeffs)
print(
    "G28212124_GLOBAL_GAUGE_COORDINATE_ATLAS_PASS =",
    G28212124_GLOBAL_GAUGE_COORDINATE_ATLAS_PASS
)


det Gram total degree = 8
det Gram term count = 15
all exponents even = True
all coefficients positive = True
G28212124_GLOBAL_GAUGE_COORDINATE_ATLAS_PASS = True



# 11. Open-neighbourhood conclusion

We have certified all reduction-regularity conditions on the **closed** box:

\[
\mathcal B=
\left\{
|K_S-1|\le10^{-2},
|\kappa_D-2|\le10^{-2},
|M_{\rm Pl}^2-1|\le10^{-2}
\right\}.
\]

Therefore its interior:

\[
\boxed{
\mathcal U=
\left\{
|K_S-1|<10^{-2},
|\kappa_D-2|<10^{-2},
|M_{\rm Pl}^2-1|<10^{-2}
\right\}
}
\]

is an explicit open coupling neighbourhood on which the principal reduction remains regular.

What has **not** yet been proved over \(\mathcal U\):

- global reality of the residual roots;
- persistence of the collision classification;
- definite type at moving collision loci;
- uniform bounded projectors over \(\mathcal U\times S^2\);
- strong hyperbolicity for every coupling in \(\mathcal U\).

Those are the next spectral-persistence locks.


In [13]:

G28212124_CLOSED_COUPLING_BOX_REDUCTION_REGULAR_CERTIFIED=all([
    G28212124_RAW_PENCIL_SYMMETRY_STRUCTURAL_PASS,
    G28212124_STRUCTURAL_KERNEL_PASS,
    G28212124_FIXED_COMPLEMENT_PASS,
    G28212124_KINETIC_RANK14_CLOSED_BOX_CERTIFIED,
    G28212124_PRINCIPAL_NOETHER_CHAIN_COUPLING_FAMILY_PASS,
    G28212124_GLOBAL_GAUGE_COORDINATE_ATLAS_PASS,
])

G28212124_OPEN_COUPLING_REDUCTION_REGULARITY_NEIGHBORHOOD_CERTIFIED=(
    G28212124_CLOSED_COUPLING_BOX_REDUCTION_REGULAR_CERTIFIED
)

G28212124_STRONG_HYPERBOLICITY_COUPLING_NEIGHBORHOOD_PROVEN=False
G28212124_BACKGROUND_NEIGHBORHOOD_OPENED=False
G28212124_GLOBAL_PARAMETER_DOMAIN_PROVEN=False

assert G28212124_CLOSED_COUPLING_BOX_REDUCTION_REGULAR_CERTIFIED
assert G28212124_OPEN_COUPLING_REDUCTION_REGULARITY_NEIGHBORHOOD_CERTIFIED

assert not G28212124_STRONG_HYPERBOLICITY_COUPLING_NEIGHBORHOOD_PROVEN
assert not G28212124_BACKGROUND_NEIGHBORHOOD_OPENED
assert not G28212124_GLOBAL_PARAMETER_DOMAIN_PROVEN

G28212124_NEXT_AUTHORIZED=(
    ".28.21.2.1.2.5 — exact/numerically-assisted coupling-neighbourhood "
    "spectral persistence and definite-type margin certificate"
)

print(
    "G28212124_OPEN_COUPLING_REDUCTION_REGULARITY_NEIGHBORHOOD_CERTIFIED =",
    G28212124_OPEN_COUPLING_REDUCTION_REGULARITY_NEIGHBORHOOD_CERTIFIED
)
print(
    "G28212124_STRONG_HYPERBOLICITY_COUPLING_NEIGHBORHOOD_PROVEN =",
    G28212124_STRONG_HYPERBOLICITY_COUPLING_NEIGHBORHOOD_PROVEN
)
print("NEXT_AUTHORIZED =",G28212124_NEXT_AUTHORIZED)


G28212124_OPEN_COUPLING_REDUCTION_REGULARITY_NEIGHBORHOOD_CERTIFIED = True
G28212124_STRONG_HYPERBOLICITY_COUPLING_NEIGHBORHOOD_PROVEN = False
NEXT_AUTHORIZED = .28.21.2.1.2.5 — exact/numerically-assisted coupling-neighbourhood spectral persistence and definite-type margin certificate



# 12. Four-level protocol

## Level 1 — GVH
The raw principal pencil comes from the same candidate action and fixed anisotropic background.

## Level 2 — exact mathematics
The coupling box is certified using exact symbolic factorisation and rational interval bounds.

## Level 3 — diagnostics
No random sampling, floating rank, SVD, or tolerance decides the PASS.

## Level 4 — observables / SI
No observable, phenomenological scale, or SI calibration is introduced.

This notebook certifies only reduction regularity on an open coupling neighbourhood.


In [14]:

ESTABLISHED_PHYSICS_USED_AS_BENCHMARK_NOT_SUBSTITUTE=True

LEVEL1_GVH_PASS=True
LEVEL2_EXACT_COUPLING_BOX_PASS=(
    G28212124_OPEN_COUPLING_REDUCTION_REGULARITY_NEIGHBORHOOD_CERTIFIED
)
LEVEL3_NO_NUMERICAL_GATE_PASS=True

UNIVERSAL_THEORY_SELECTED_SI_SCALE_RANK=0
NUMERICAL_SI_CALIBRATION_AUTHORIZED=False
LEVEL4_SI_LEDGER_PASS=True

FOUR_LEVEL_PROTOCOL_PASS=all([
    LEVEL1_GVH_PASS,
    LEVEL2_EXACT_COUPLING_BOX_PASS,
    LEVEL3_NO_NUMERICAL_GATE_PASS,
    LEVEL4_SI_LEDGER_PASS,
])

assert FOUR_LEVEL_PROTOCOL_PASS

print("FOUR_LEVEL_PROTOCOL_PASS =",FOUR_LEVEL_PROTOCOL_PASS)


FOUR_LEVEL_PROTOCOL_PASS = True


In [15]:

verdict={
    "notebook":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.4_"
        "Exact_Open_Coupling_Neighborhood_Reduction_Regularity_FAST",
    "parent_B3":PARENT_B3,
    "scope":{
        "background":{
            "a0":"3/4",
            "a1":"-1/5",
            "a2":"-1/4",
            "a3":"-3/10",
            "fixed":True,
        },
        "coupling_center":{
            "K_S":"1",
            "kappa_D":"2",
            "Mpl2":"1",
        },
        "closed_box_radius":"1/100",
        "open_interior_certified":True,
    },
    "exact":{
        "raw_pencil_symmetry_structural_pass":
            bool(G28212124_RAW_PENCIL_SYMMETRY_STRUCTURAL_PASS),
        "structural_kernel_pass":
            bool(G28212124_STRUCTURAL_KERNEL_PASS),
        "fixed_complement_pass":
            bool(G28212124_FIXED_COMPLEMENT_PASS),
        "kinetic_determinant_factorization_pass":
            bool(G28212124_KINETIC_DETERMINANT_FACTORIZATION_PASS),
        "all_affine_factor_margins_pass":
            bool(G28212124_ALL_AFFINE_FACTOR_MARGINS_PASS),
        "cubic_factor_margin_pass":
            bool(G28212124_CUBIC_FACTOR_MARGIN_PASS),
        "kinetic_rank14_closed_box_certified":
            bool(G28212124_KINETIC_RANK14_CLOSED_BOX_CERTIFIED),
        "principal_noether_chain_coupling_family_pass":
            bool(G28212124_PRINCIPAL_NOETHER_CHAIN_COUPLING_FAMILY_PASS),
        "global_gauge_coordinate_atlas_pass":
            bool(G28212124_GLOBAL_GAUGE_COORDINATE_ATLAS_PASS),
        "closed_coupling_box_reduction_regular_certified":
            bool(G28212124_CLOSED_COUPLING_BOX_REDUCTION_REGULAR_CERTIFIED),
        "open_coupling_reduction_regularity_neighborhood_certified":
            bool(G28212124_OPEN_COUPLING_REDUCTION_REGULARITY_NEIGHBORHOOD_CERTIFIED),
    },
    "locks":{
        "strong_hyperbolicity_coupling_neighborhood_proven":
            bool(G28212124_STRONG_HYPERBOLICITY_COUPLING_NEIGHBORHOOD_PROVEN),
        "background_neighborhood_opened":
            bool(G28212124_BACKGROUND_NEIGHBORHOOD_OPENED),
        "global_parameter_domain_proven":
            bool(G28212124_GLOBAL_PARAMETER_DOMAIN_PROVEN),
    },
    "protocol":{
        "four_level_protocol_pass":
            bool(FOUR_LEVEL_PROTOCOL_PASS),
        "universal_theory_selected_SI_scale_rank":0,
    },
    "status":
        "PASS_EXACT_OPEN_COUPLING_REDUCTION_REGULARITY_"
        "SPECTRAL_PERSISTENCE_OPEN",
    "next_authorized":
        G28212124_NEXT_AUTHORIZED,
}

export_dir=Path("/mnt/data/gvh_exports_28212124")
export_dir.mkdir(parents=True,exist_ok=True)

verdict_path=export_dir / (
    "gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.4_"
    "Exact_Open_Coupling_Neighborhood_Reduction_Regularity_FAST.json"
)

verdict_path.write_text(
    json.dumps(
        verdict,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("STATUS =",verdict["status"])
print(
    "OPEN_COUPLING_REDUCTION_REGULARITY_NEIGHBORHOOD_CERTIFIED =",
    verdict["exact"][
        "open_coupling_reduction_regularity_neighborhood_certified"
    ]
)
print(
    "STRONG_HYPERBOLICITY_COUPLING_NEIGHBORHOOD_PROVEN =",
    verdict["locks"][
        "strong_hyperbolicity_coupling_neighborhood_proven"
    ]
)
print("verdict JSON =",verdict_path)


STATUS = PASS_EXACT_OPEN_COUPLING_REDUCTION_REGULARITY_SPECTRAL_PERSISTENCE_OPEN
OPEN_COUPLING_REDUCTION_REGULARITY_NEIGHBORHOOD_CERTIFIED = True
STRONG_HYPERBOLICITY_COUPLING_NEIGHBORHOOD_PROVEN = False
verdict JSON = /mnt/data/gvh_exports_28212124/gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.4_Exact_Open_Coupling_Neighborhood_Reduction_Regularity_FAST.json
